# ConformerFlow — Google Colab Notebook

**NMR-trained SE(3) Flow Matching for Protein Conformational Ensembles**

This notebook runs the full ConformerFlow pipeline on Colab:
1. Install dependencies
2. Mount Google Drive (for persistent storage)
3. Set up the code and data
4. Train the model
5. Generate conformational ensembles
6. Evaluate against NMR ground truth
7. Run baseline comparisons

**Recommended runtime:** GPU → T4 (free) or A100 (Colab Pro)  
**Required disk:** ~10 GB (PDB data + checkpoints)  
**Training time (default config):** ~4 h on T4 / ~1.5 h on A100

> ⚡ **Before starting**: Runtime → Change runtime type → GPU

## 0 · Check GPU

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    print('❌  No GPU detected. Go to Runtime → Change runtime type → GPU')
else:
    # Extract GPU name and VRAM
    for line in gpu_info.stdout.splitlines():
        if 'MiB' in line or 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line:
            print(line)
    print('\n✅  GPU ready')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 1 · Mount Google Drive
All checkpoints, PDB data, and results are saved here so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── Change this if you want a different Drive folder ──────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/conformerflow')
# ─────────────────────────────────────────────────────────────────────

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

PDB_DATA_DIR  = DRIVE_ROOT / 'pdb_data'
CHECKPOINT_DIR= DRIVE_ROOT / 'checkpoints'
RESULTS_DIR   = DRIVE_ROOT / 'results'

for d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Drive root : {DRIVE_ROOT}')
print(f'PDB data   : {PDB_DATA_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'Results    : {RESULTS_DIR}')

## 2 · Install Dependencies

In [ ]:
# Core required packages (torch, numpy, scipy, requests, tqdm are pre-installed on Colab)
!pip install -q biopython gemmi einops

# Optional: ESM-2 sequence embeddings (adds ~1.5 GB; skip if on free T4)
INSTALL_ESM2 = False   # set True if you have enough VRAM (A100 recommended)
if INSTALL_ESM2:
    !pip install -q fair-esm

# Optional: Weights & Biases training dashboard
INSTALL_WANDB = False
if INSTALL_WANDB:
    !pip install -q wandb

print('\n✅  Required packages installed')

In [ ]:
# ── OpenMM (energy minimization after generation) ─────────────────────
# Uses condacolab to get conda on Colab, then installs OpenMM.
# Skip this cell if you don't need energy minimization.
INSTALL_OPENMM = False   # set True to enable --minimize in validation

if INSTALL_OPENMM:
    !pip install -q condacolab
    import condacolab
    condacolab.install()   # this restarts the kernel — re-run from Cell 0 after
    # After kernel restart, run:
    # !conda install -c conda-forge openmm -y -q
else:
    print('OpenMM skipped — energy minimization will be disabled')
    print('Set INSTALL_OPENMM = True and re-run to enable it')

In [ ]:
# ── MMseqs2 (sequence redundancy removal) ─────────────────────────────
# Binary install. Without it, filter.py uses the Python fallback (slower, approximate).
INSTALL_MMSEQS2 = False   # set True for publication-quality sequence clustering

if INSTALL_MMSEQS2:
    !wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
    !tar xf mmseqs-linux-avx2.tar.gz
    import os
    os.environ['PATH'] = '/content/mmseqs/bin:' + os.environ['PATH']
    !mmseqs version
    print('✅  MMseqs2 ready')
else:
    print('MMseqs2 skipped — Python pairwise identity fallback will be used')

## 3 · Set Up Code

In [ ]:
from pathlib import Path
import sys, subprocess, shutil

# ── Repo URL — change only if you forked the project ──────────────────
GITHUB_REPO = 'https://github.com/kumar23kan/conformerflow.git'
CLONE_DIR   = Path('/content/conformerflow')
CODE_DIR    = CLONE_DIR / 'files'
# ──────────────────────────────────────────────────────────────────────

def _do_clone():
    """Fresh clone into CLONE_DIR."""
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    print(f'Cloning {GITHUB_REPO} …')
    r = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl', GITHUB_REPO, str(CLONE_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            f'git clone failed (return code {r.returncode}):\n{r.stderr}\n\n'
            'If the repo is private, use Option B below to upload a zip instead.'
        )
    print('✅  Cloned successfully')

if not CLONE_DIR.exists():
    # First run on this Colab session
    _do_clone()
elif not CODE_DIR.exists():
    # Stale clone that pre-dates the files/ directory — re-clone
    print('⚠️  Stale clone detected (files/ missing) — re-cloning …')
    _do_clone()
else:
    # Repo already present — pull latest changes
    r = subprocess.run(
        ['git', '-C', str(CLONE_DIR), 'pull', '--ff-only'],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f'✅  Repo already present at {CLONE_DIR} (pulled latest)')
    else:
        print(f'⚠️  git pull failed — using existing clone\n{r.stderr}')

if not CODE_DIR.exists():
    raise RuntimeError(
        f'Setup failed: {CODE_DIR} still does not exist.\n'
        'Check that the GitHub repo is accessible and try Option B below.'
    )

sys.path.insert(0, str(CODE_DIR))
print(f'Code directory : {CODE_DIR}')
print(f'Exists         : {CODE_DIR.exists()}')
print('Contents       :', sorted([p.name for p in CODE_DIR.iterdir()]))

# ── Option B: upload a zip instead of cloning ─────────────────────────
# If the repo is private or git clone fails, upload a zip manually:
#
#   from google.colab import files
#   uploaded = files.upload()          # upload conformerflow.zip
#   import zipfile, io
#   z = zipfile.ZipFile(io.BytesIO(list(uploaded.values())[0]))
#   z.extractall('/content/')
#   CODE_DIR = Path('/content/conformerflow/files')
#   sys.path.insert(0, str(CODE_DIR))

## 4 · Configuration
Edit these variables to control the full run — no config file editing needed.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURATION — edit these before running the pipeline
# ══════════════════════════════════════════════════════════════

# ── Training mode ──────────────────────────────────────────────
# 'flow_matching' | 'ot_cfm' | 'ddpm' | 'ddim' | 'vae' |
# 'score_matching' | 'multi_head'
TRAINING_MODE   = 'flow_matching'

# ── Structure encoder ──────────────────────────────────────────
# 'se3_frames' | 'cartesian' | 'distance_matrix' | 'torsion_angles'
STRUCT_REPR     = 'se3_frames'

# ── Latent type (P1-2 ablation) ───────────────────────────────
# 'per_residue' | 'global'
LATENT_TYPE     = 'per_residue'

# ── Model size ─────────────────────────────────────────────────
# T4 (15 GB VRAM) — safe defaults below
# A100 (40 GB): d_model=512, max_residues=800, batch_size=8
D_MODEL         = 128    # 128 for T4; 256+ for A100
MAX_RESIDUES    = 256    # 256 for T4; 512+ for A100

# ── Training ───────────────────────────────────────────────────
BATCH_SIZE      = 1      # 1 for T4; 4–8 for A100
MAX_EPOCHS      = 10     # 10 for smoke test; 100 for publication
LEARNING_RATE   = 1e-4
N_GEN_TRAIN     = 4      # conformers per step; reduce to 2 if still OOM
USE_BF16        = False  # True on A100 for ~2× speedup

# ── Inference ──────────────────────────────────────────────────
N_CONFORMERS    = 20     # conformers to generate at inference
N_STEPS         = 20     # ODE integration steps
ODE_METHOD      = 'heun' # 'heun' | 'euler' | 'rk4'

# ── Data ───────────────────────────────────────────────────────
# Number of NMR structures to download (reduce for quick tests)
N_STRUCTURES    = 500    # set None to download full dataset (~10k structures)

# ── Sequence encoder ───────────────────────────────────────────
SEQ_ENCODER = 'esm2_650m' if INSTALL_ESM2 else 'onehot'

# ── Energy minimization (requires OpenMM) ─────────────────────
USE_MINIMIZE = INSTALL_OPENMM

# ── WandB run name ─────────────────────────────────────────────
RUN_NAME = f'colab_{TRAINING_MODE}_{STRUCT_REPR}'

print('Configuration:')
for k, v in [
    ('training_mode', TRAINING_MODE), ('struct_repr', STRUCT_REPR),
    ('latent_type', LATENT_TYPE), ('d_model', D_MODEL),
    ('max_residues', MAX_RESIDUES), ('batch_size', BATCH_SIZE),
    ('max_epochs', MAX_EPOCHS), ('seq_encoder', SEQ_ENCODER),
    ('n_conformers', N_CONFORMERS), ('minimize', USE_MINIMIZE),
]:
    print(f'  {k:20s}: {v}')

## 5 · Download PDB Data
Downloads NMR + paired X-ray structures from RCSB. Skips files already on Drive.

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
from data.fetch_pdb import fetch_nmr_ids, download_pdbs

NMR_DIR  = PDB_DATA_DIR / 'nmr'
XRAY_DIR = PDB_DATA_DIR / 'xray'
NMR_DIR.mkdir(exist_ok=True)
XRAY_DIR.mkdir(exist_ok=True)

# Fetch list of NMR structure IDs from RCSB
if 'N_STRUCTURES' not in globals():
    N_STRUCTURES = 500  # default: 500 structures for quick start
print('Fetching NMR structure IDs from RCSB...')
nmr_ids = fetch_nmr_ids(max_results=N_STRUCTURES)
print(f'Found {len(nmr_ids):,} NMR entries')

# Download (skips already-downloaded files)
print(f'Downloading to {NMR_DIR} ...')
download_pdbs(nmr_ids, output_dir=str(NMR_DIR), max_workers=4)
nmr_files = list(NMR_DIR.glob('*.pdb'))
print(f'✅  {len(nmr_files):,} NMR PDB files ready')

## 6 · Parse NMR Ensembles

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
from data.parse_nmr import parse_nmr_pdb, ensemble_to_tensors
import numpy as np
from tqdm.notebook import tqdm

PARSED_DIR = PDB_DATA_DIR / 'parsed'
PARSED_DIR.mkdir(exist_ok=True)

n_ok, n_fail, n_skip = 0, 0, 0
pdb_files = sorted(NMR_DIR.glob('*.pdb'))

for pdb_path in tqdm(pdb_files, desc='Parsing NMR'):
    npz_path = PARSED_DIR / f'{pdb_path.stem}.npz'
    if npz_path.exists():
        n_skip += 1
        continue

    try:
        ensemble = parse_nmr_pdb(str(pdb_path))
        if ensemble is None:
            n_fail += 1
            continue
        tensors = ensemble_to_tensors(ensemble)
        np.savez_compressed(
            str(npz_path),
            coords   = tensors['coords'],
            one_hot  = tensors['one_hot'],
            mask     = tensors['mask'],
        )
        # Save metadata alongside
        import json
        with open(npz_path.with_suffix('.json'), 'w') as f:
            json.dump({
                'pdb_id':           pdb_path.stem,
                'sequence':         ensemble.sequence,
                'n_conformers':     tensors['coords'].shape[0],
                'n_residues':       tensors['coords'].shape[1],
                'deposition_date':  getattr(ensemble, 'deposition_date', None),
            }, f)
        n_ok += 1
    except Exception as e:
        n_fail += 1

print(f'\nParsed: {n_ok} ok | {n_fail} failed | {n_skip} skipped (cached)')
print(f'Total .npz files: {len(list(PARSED_DIR.glob("*.npz")))}')

## 7 · Filter & Split Dataset

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
from data.filter import run_filter_pipeline

SPLITS_DIR = PDB_DATA_DIR / 'splits'

splits_info = run_filter_pipeline(
    parsed_dir          = str(PARSED_DIR),
    output_dir          = str(SPLITS_DIR),
    cluster_identity    = 0.30,    # 30% sequence identity cutoff
    use_temporal_split  = True,    # entries deposited >= 2020 → test set
    test_cutoff_year    = 2020,
    min_conformers      = 5,
    min_residues        = 20,
    max_residues        = MAX_RESIDUES,
    min_spread_rmsd     = 0.1,
)

for split, items in splits_info['splits'].items():
    print(f'{split:5s}: {len(items):4,} entries')
print(f"\nTotal passed quality filters: {splits_info['total_passed']:,}")

## 8 · Build Config & Train
Writes a resolved config dict and launches training. Progress is logged to the cell output.

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
from configs.config_validator import load_and_validate, set_nested

# Load base config then apply all our Colab settings
cfg = load_and_validate(
    str(CODE_DIR / 'configs' / 'base_config.yaml'),
    auto_batch=False,
    verbose=False,
)

overrides = [
    ('training_mode',                      TRAINING_MODE),
    ('representation.structure',           STRUCT_REPR),
    ('representation.sequence',            SEQ_ENCODER),
    ('ensemble_stats.latent_type',         LATENT_TYPE),
    ('model.d_model',                      D_MODEL),
    ('data.max_residues',                  MAX_RESIDUES),
    ('data.pdb_data_dir',                  str(PDB_DATA_DIR)),
    ('training.batch_size',                BATCH_SIZE),
    ('training.max_epochs',                MAX_EPOCHS),
    ('training.learning_rate',             LEARNING_RATE),
    ('training.n_gen_conformers',          N_GEN_TRAIN),
    ('training.checkpoint_dir',            str(CHECKPOINT_DIR)),
    ('training.use_bf16',                  USE_BF16),
    ('training.multi_gpu',                 False),
    ('logging.run_name',                   RUN_NAME),
    ('logging.use_wandb',                  INSTALL_WANDB),
    ('generative_model.n_inference_steps', N_STEPS),
    ('generative_model.ode_method',        ODE_METHOD),
]

for path, value in overrides:
    set_nested(cfg, path, value)

print('Config summary:')
for path, value in overrides:
    print(f'  {path:42s}: {value}')

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
import json as _json
from pathlib import Path as _Path
from data.dataset     import build_dataloaders
from training.trainer import Trainer
from configs.config_validator import load_and_validate, set_nested

# ── Resolve all paths locally (safe if prior cells were skipped) ──────
_DRIVE_ROOT    = _Path('/content/drive/MyDrive/conformerflow')
PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
SPLITS_DIR     = PDB_DATA_DIR / 'splits'
TRAIN_MANIFEST = SPLITS_DIR / 'train.json'
VAL_MANIFEST   = SPLITS_DIR / 'val.json'

# ── Fall back to config defaults if variables from Cell 4/12 are absent ──
_BATCH_SIZE      = globals().get('BATCH_SIZE',      2)
_MAX_RESIDUES    = globals().get('MAX_RESIDUES',    512)
_N_GEN_TRAIN     = globals().get('N_GEN_TRAIN',     8)
_MAX_EPOCHS      = globals().get('MAX_EPOCHS',      50)
_LEARNING_RATE   = globals().get('LEARNING_RATE',   1e-4)
_TRAINING_MODE   = globals().get('TRAINING_MODE',   'flow_matching')
_STRUCT_REPR     = globals().get('STRUCT_REPR',     'se3_frames')
_LATENT_TYPE     = globals().get('LATENT_TYPE',     'per_residue')
_D_MODEL         = globals().get('D_MODEL',         256)
_USE_BF16        = globals().get('USE_BF16',        False)
_RUN_NAME        = globals().get('RUN_NAME',        'colab_run')
_N_STEPS         = globals().get('N_STEPS',         20)
_ODE_METHOD      = globals().get('ODE_METHOD',      'heun')
_SEQ_ENCODER     = globals().get('SEQ_ENCODER',     'onehot')
_INSTALL_WANDB   = globals().get('INSTALL_WANDB',   False)

# ── Pre-flight: make sure the data pipeline has run ───────────────────
def _count(path):
    if not _Path(path).exists():
        return 0
    try:
        return len(_json.load(open(path)))
    except Exception:
        return 0

n_train = _count(TRAIN_MANIFEST)
n_val   = _count(VAL_MANIFEST)

if n_train == 0:
    raise RuntimeError(
        "\n\n❌  No training samples found — data pipeline has not completed.\n\n"
        "   Run these cells IN ORDER first:\n"
        "     Cell 2  → Mount Google Drive\n"
        "     Cell 5  → Download PDB Data  (fetches NMR structures from RCSB)\n"
        "     Cell 6  → Parse NMR Ensembles\n"
        "     Cell 7  → Filter & Split Dataset\n"
        "     Cell 8  → Build Config\n"
        "   Then re-run this cell.\n"
        f"\n   Expected manifest: {TRAIN_MANIFEST}"
    )

print(f"Manifests ready — train: {n_train:,}  val: {n_val:,}")

# ── Rebuild cfg (safe even if Cell 8 was skipped) ─────────────────────
if 'cfg' not in globals():
    cfg = load_and_validate(
        str(CODE_DIR / 'configs' / 'base_config.yaml'),
        auto_batch=False, verbose=False,
    )
    for _path, _val in [
        ('training_mode',                      _TRAINING_MODE),
        ('representation.structure',           _STRUCT_REPR),
        ('representation.sequence',            _SEQ_ENCODER),
        ('ensemble_stats.latent_type',         _LATENT_TYPE),
        ('model.d_model',                      _D_MODEL),
        ('data.max_residues',                  _MAX_RESIDUES),
        ('data.pdb_data_dir',                  str(PDB_DATA_DIR)),
        ('training.batch_size',                _BATCH_SIZE),
        ('training.max_epochs',                _MAX_EPOCHS),
        ('training.learning_rate',             _LEARNING_RATE),
        ('training.n_gen_conformers',          _N_GEN_TRAIN),
        ('training.checkpoint_dir',            str(CHECKPOINT_DIR)),
        ('training.use_bf16',                  _USE_BF16),
        ('training.multi_gpu',                 False),
        ('logging.run_name',                   _RUN_NAME),
        ('logging.use_wandb',                  _INSTALL_WANDB),
        ('generative_model.n_inference_steps', _N_STEPS),
        ('generative_model.ode_method',        _ODE_METHOD),
    ]:
        set_nested(cfg, _path, _val)
    print('Config rebuilt from defaults (Cell 8 was skipped)')

# ── Build data loaders ────────────────────────────────────────────────
print('Building data loaders...')
loaders = build_dataloaders(
    train_manifest = str(TRAIN_MANIFEST),
    val_manifest   = str(VAL_MANIFEST),
    test_manifest  = str(VAL_MANIFEST),
    batch_size     = _BATCH_SIZE,
    max_residues   = _MAX_RESIDUES,
    max_conformers = _N_GEN_TRAIN,
    num_workers    = 2,
)
print(f"Train batches: {len(loaders['train'])}  |  Val batches: {len(loaders['val'])}")

# ── Train ─────────────────────────────────────────────────────────────
print('\nBuilding model and starting training...')
print('─' * 60)
trainer = Trainer(cfg)
trainer.train(loaders['train'], loaders['val'])
print('─' * 60)
print('✅  Training complete')
print(f'Best checkpoint: {CHECKPOINT_DIR}/ckpt_best.pt')


## 9 · Generate a Conformational Ensemble
Feed a single PDB file (X-ray or NMR) and generate N predicted conformers.

In [ ]:
# ── Input structure ────────────────────────────────────────────────────
# Option A: use one of the downloaded PDB files
example_pdb = sorted(NMR_DIR.glob('*.pdb'))[0]

# Option B: upload your own PDB
# from google.colab import files
# uploaded = files.upload()
# example_pdb = Path(list(uploaded.keys())[0])

print(f'Input PDB: {example_pdb}')

# ── Run inference ─────────────────────────────────────────────────────
from inference.predict import ConformerFlowPredictor

BEST_CKPT = str(CHECKPOINT_DIR / 'ckpt_best.pt')

predictor = ConformerFlowPredictor(BEST_CKPT, device='cuda')
result    = predictor.predict(
    pdb_path     = str(example_pdb),
    n_conformers = N_CONFORMERS,
    n_steps      = N_STEPS,
    method       = ODE_METHOD,
)

print(f'\nPrediction complete:')
print(f'  Protein      : {result.pdb_id}')
print(f'  Sequence len : {result.n_residues} residues')
print(f'  Conformers   : {result.n_conformers}')
print(f'  CA coords    : {result.ca_coords.shape}  (conformers × residues × 3)')

import numpy as np
# Quick diversity check: mean pairwise CA-RMSD across generated ensemble
ca = result.ca_coords
rmsds = []
for i in range(min(10, len(ca))):
    for j in range(i+1, min(10, len(ca))):
        diff = ca[i] - ca[j]
        rmsds.append(np.sqrt((diff**2).sum(axis=-1).mean()))
print(f'  Mean pairwise RMSD: {np.mean(rmsds):.2f} Å  (diversity check)')

In [ ]:
# ── Optional: save ensemble as multi-model PDB ───────────────────────
ENSEMBLE_PDB = RESULTS_DIR / f'{result.pdb_id}_ensemble.pdb'

with open(ENSEMBLE_PDB, 'w') as f:
    for model_idx, coords in enumerate(result.ca_coords):
        f.write(f'MODEL     {model_idx+1:4d}\n')
        for res_idx, (x, y, z) in enumerate(coords):
            aa = result.sequence[res_idx] if res_idx < len(result.sequence) else 'X'
            f.write(
                f'ATOM  {res_idx+1:5d}  CA  ALA A{res_idx+1:4d}    '
                f'{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n'
            )
        f.write('ENDMDL\n')
    f.write('END\n')

print(f'✅  Saved multi-model PDB: {ENSEMBLE_PDB}')

# Download to local machine
from google.colab import files
files.download(str(ENSEMBLE_PDB))

## 10 · Visualise the Ensemble
Per-residue RMSF profile plotted with matplotlib.

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 120})

from evaluation.metrics import compute_rmsf

rmsf_pred = compute_rmsf(result.ca_coords)   # (L,)
L         = result.n_residues
res_ids   = range(1, L + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# RMSF profile
ax = axes[0]
ax.fill_between(res_ids, rmsf_pred, alpha=0.3, color='steelblue')
ax.plot(res_ids, rmsf_pred, color='steelblue', linewidth=1.2)
ax.set_xlabel('Residue')
ax.set_ylabel('RMSF (Å)')
ax.set_title(f'{result.pdb_id} — Per-residue CA flexibility')
ax.set_xlim(1, L)
ax.grid(alpha=0.3)

# Radius of gyration distribution
from evaluation.metrics import compute_rg_distribution
rg = compute_rg_distribution(result.ca_coords)
ax = axes[1]
ax.hist(rg, bins=10, color='coral', edgecolor='white', alpha=0.85)
ax.axvline(rg.mean(), color='red', linestyle='--', label=f'mean = {rg.mean():.1f} Å')
ax.set_xlabel('Radius of gyration (Å)')
ax.set_ylabel('Count')
ax.set_title('Ensemble compactness distribution')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / f'{result.pdb_id}_rmsf.png'), bbox_inches='tight')
plt.show()
print(f'Figure saved to Drive')

## 11 · Validate Against NMR Ground Truth
Needs a protein that has both an X-ray and an NMR structure in the PDB.
The validator generates an ensemble from the X-ray structure and compares it to the deposited NMR ensemble.

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
from evaluation.validate import ConformerFlowValidator
import json

# ── Provide a paired X-ray / NMR pair ─────────────────────────────────
# Example: 1UBQ (X-ray ubiquitin) vs 1D3Z (NMR ubiquitin)
# Download them first if not already present:

XRAY_PDB_PATH = str(NMR_DIR / '1UBQ.pdb')   # change to your X-ray PDB path
NMR_PDB_PATH  = str(NMR_DIR / '1D3Z.pdb')   # change to your NMR PDB path

# Quick download if files don't exist
# RCSB serves .pdb.gz (compressed) — fetch and decompress in memory
import gzip, io, requests
for pdb_id, path in [('1UBQ', XRAY_PDB_PATH), ('1D3Z', NMR_PDB_PATH)]:
    if not Path(path).exists():
        url  = f'https://files.rcsb.org/download/{pdb_id}.pdb.gz'
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with gzip.open(io.BytesIO(resp.content)) as gz:
            data = gz.read()
        Path(path).write_bytes(data)
        print(f'Downloaded {pdb_id}')

# ── Run validation ─────────────────────────────────────────────────────
validator = ConformerFlowValidator(
    checkpoint_path = BEST_CKPT,
    n_conformers    = N_CONFORMERS,
    n_steps         = N_STEPS,
    method          = ODE_METHOD,
    device          = 'cuda',
    minimize        = USE_MINIMIZE,
)

metrics = validator.validate_one(
    xray_pdb = XRAY_PDB_PATH,
    nmr_pdb  = NMR_PDB_PATH,
    pdb_id   = '1UBQ_vs_1D3Z',
)

# Print key metrics
print('\n── Validation Results ──────────────────────────────')
key_metrics = [
    ('coverage_rmsd',           'Coverage RMSD (Å)',    True),
    ('precision_rmsd',          'Precision RMSD (Å)',   True),
    ('rmsf_pearson_r',          'RMSF Pearson r',       False),
    ('torsion_overlap',         'Torsion overlap',      False),
    ('covariance_frobenius_sim','Covariance sim',       False),
    ('clash_score_per100',      'Clash score/100 res',  True),
    ('rama_pred_favored',       'Rama favored (pred)',  False),
    ('rama_true_favored',       'Rama favored (NMR)',   False),
    ('nmr_recall_2A',           'NMR recall @ 2 Å',    False),
]
for key, label, lower_better in key_metrics:
    val = metrics.get(key)
    if val is not None:
        arrow = '↓' if lower_better else '↑'
        print(f'  {label:30s}: {val:.4f}  {arrow}')

# Save metrics
metrics_path = RESULTS_DIR / '1UBQ_vs_1D3Z_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump({k: v for k, v in metrics.items() if not hasattr(v, 'tolist') or True},
              f, indent=2, default=lambda x: x.tolist() if hasattr(x, 'tolist') else str(x))
print(f'\nFull metrics saved: {metrics_path}')

## 12 · Baseline Comparison
Compare ConformerFlow against the three baselines on the same protein.

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
from data.parse_nmr  import parse_nmr_pdb, ensemble_to_tensors
from data.parse_xray import parse_xray_pdb
from evaluation.baselines import evaluate_baselines
import numpy as np

# Load X-ray CA structure (single conformer)
xray_data = parse_xray_pdb(XRAY_PDB_PATH)
xray_ca   = xray_data['coords'][0, :, 1, :]   # (L, 3)

# Load NMR CA ensemble (ground truth)
nmr_ens  = parse_nmr_pdb(NMR_PDB_PATH)
nmr_tens = ensemble_to_tensors(nmr_ens)
nmr_ca   = nmr_tens['coords'][:, :, 1, :]     # (M, L, 3)

L = min(xray_ca.shape[0], nmr_ca.shape[1])
xray_ca = xray_ca[:L]
nmr_ca  = nmr_ca[:, :L]

print(f'X-ray: {xray_ca.shape}  |  NMR: {nmr_ca.shape}')

# Run all three baselines
print('Running baselines...')
baseline_results = evaluate_baselines(
    xray_ca      = xray_ca,
    nmr_ca       = nmr_ca,
    n_conformers = N_CONFORMERS,
    pdb_id       = '1UBQ_vs_1D3Z',
)

# Print comparison table
col_metrics = ['coverage_rmsd', 'rmsf_pearson_r', 'torsion_overlap',
               'covariance_frobenius_sim', 'clash_score_per100', 'rama_pred_favored']
header = f"{'Method':22s}" + ''.join(f"{m[:12]:>14s}" for m in col_metrics)
print('\n' + '─'*len(header))
print(header)
print('─'*len(header))

# ConformerFlow row (from cell 11)
cf_vals = ''.join(f"{metrics.get(m, float('nan')):14.4f}" for m in col_metrics)
print(f"{'ConformerFlow':22s}{cf_vals}  ← our model")

for bname, bmet in baseline_results.items():
    row = ''.join(f"{bmet.get(m, float('nan')):14.4f}" for m in col_metrics)
    print(f"{bname:22s}{row}")

print('─'*len(header))
print('↓ = lower is better   ↑ = higher is better')
print('coverage_rmsd↓  rmsf_pearson_r↑  torsion_overlap↑  cov_sim↑  clash↓  rama_fav↑')

In [ ]:
# ── Path guard: ensures CODE_DIR is on sys.path even if Cell 3 was skipped ──
import sys
from pathlib import Path
_CLONE_DIR = Path('/content/conformerflow')
_CODE_DIR  = _CLONE_DIR / 'files'
if not _CODE_DIR.exists():
    import subprocess, shutil
    if _CLONE_DIR.exists():
        shutil.rmtree(_CLONE_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'claude/document-conformerflow-anGHl',
         'https://github.com/kumar23kan/conformerflow.git', str(_CLONE_DIR)],
        check=True
    )
CODE_DIR = _CODE_DIR
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# ─────────────────────────────────────────────────────────────────────────────
# ── Path fallback: define storage paths if Cell 2 was skipped ────────
from pathlib import Path as _P
if 'PDB_DATA_DIR' not in globals():
    _drive = _P('/content/drive/MyDrive')
    if _drive.exists():
        _DRIVE_ROOT = _drive / 'conformerflow'
        print('Using Google Drive storage')
    else:
        _DRIVE_ROOT = _P('/content/conformerflow_data')
        print('Drive not mounted — using local /content storage (data lost on session end)')
    PDB_DATA_DIR   = _DRIVE_ROOT / 'pdb_data'
    CHECKPOINT_DIR = _DRIVE_ROOT / 'checkpoints'
    RESULTS_DIR    = _DRIVE_ROOT / 'results'
    PARSED_DIR     = PDB_DATA_DIR / 'parsed'
    SPLITS_DIR     = PDB_DATA_DIR / 'splits'
    for _d in [PDB_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, PARSED_DIR, SPLITS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────
# ── Plot RMSF comparison ───────────────────────────────────────────────
from evaluation.metrics import compute_rmsf
from evaluation.baselines import StaticBaseline, GaussianNoiseBaseline, NMRMeanBaseline
import matplotlib.pyplot as plt

rmsf_nmr    = compute_rmsf(nmr_ca)
rmsf_cf     = compute_rmsf(result.ca_coords[:, :L])
rmsf_static = compute_rmsf(StaticBaseline().generate(xray_ca, N_CONFORMERS))
rmsf_gauss  = compute_rmsf(GaussianNoiseBaseline().generate(xray_ca, N_CONFORMERS, nmr_ca))

fig, ax = plt.subplots(figsize=(12, 4))
x = range(1, L + 1)
ax.plot(x, rmsf_nmr,    color='black',     linewidth=2,   label='NMR ground truth',  zorder=5)
ax.plot(x, rmsf_cf,     color='steelblue', linewidth=1.8, label='ConformerFlow',     zorder=4)
ax.plot(x, rmsf_gauss,  color='coral',     linewidth=1.2, label='Gaussian baseline', zorder=3, linestyle='--')
ax.fill_between(x, rmsf_static, alpha=0.15, color='gray',  label='Static baseline')

ax.set_xlabel('Residue')
ax.set_ylabel('RMSF (Å)')
ax.set_title('Per-residue flexibility: ConformerFlow vs baselines')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(1, L)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'rmsf_comparison.png'), bbox_inches='tight', dpi=150)
plt.show()

## 13 · Full Dataset Validation (Optional)
Runs on all held-out pairs and prints the aggregated report. Can take 30–60 min on T4.

In [ ]:
# Create a paired.json mapping NMR IDs to X-ray IDs
# In a real pipeline this comes from your data preparation step.
# Here we use the test split entries and assume xray_id == pdb_id for demonstration.

import json
from pathlib import Path

test_manifest = json.loads((SPLITS_DIR / 'test.json').read_text())
# Build a pairs list: each entry needs 'nmr_id' and 'xray_id'
# Adjust this to match your actual paired data:
pairs = [{'nmr_id': e['pdb_id'], 'xray_id': e['pdb_id']} for e in test_manifest[:50]]

PAIRS_JSON = RESULTS_DIR / 'test_pairs.json'
PAIRS_JSON.write_text(json.dumps(pairs, indent=2))
print(f'{len(pairs)} pairs written to {PAIRS_JSON}')

In [ ]:
VALIDATION_OUT = RESULTS_DIR / 'validation'

summary = validator.validate_dataset(
    pairs_json    = str(PAIRS_JSON),
    xray_dir      = str(NMR_DIR),   # using NMR files as proxy; swap for real X-ray dir
    nmr_dir       = str(NMR_DIR),
    output_dir    = str(VALIDATION_OUT),
    max_proteins  = 50,             # set None to run all
)

print(f"\nSuccess rate: {summary['n_success']}/{summary['n_total']}")

## 14 · Download All Results
Everything is already on Drive. Use the cells below to also download to your local machine.

In [ ]:
# Zip results directory and download
import shutil
from google.colab import files

zip_path = '/content/conformerflow_results.zip'
shutil.make_archive('/content/conformerflow_results', 'zip', str(RESULTS_DIR))
print(f'Zipped results: {zip_path}')
files.download(zip_path)

In [ ]:
# Download best checkpoint
from google.colab import files
ckpt = CHECKPOINT_DIR / 'ckpt_best.pt'
if ckpt.exists():
    files.download(str(ckpt))
    print(f'Downloaded: {ckpt.name}')
else:
    print('No checkpoint found yet — run training first')

---
## Tips & Troubleshooting

| Problem | Fix |
|---------|-----|
| CUDA out of memory | Reduce `BATCH_SIZE` to 1, `D_MODEL` to 128, `MAX_RESIDUES` to 256 |
| Session disconnects during training | Checkpoints are on Drive — resume with `training.resume_from = str(CHECKPOINT_DIR / 'ckpt_latest.pt')` |
| `import` errors | Re-run the config cell (Cell 4) to restore `sys.path` after kernel restart |
| OpenMM not found | Skip `INSTALL_OPENMM = True` — energy minimization is optional |
| Slow parsing | Reduce `N_STRUCTURES` to 100 for a quick smoke test |
| Want A100 | Colab Pro → Runtime → Change runtime type → A100 GPU |

### Resume after disconnection
All parsed data and checkpoints are on Drive. After reconnection:
1. Run cells 0–4 (GPU check, Drive mount, installs, config)
2. Skip to the cell you need — parsing and filtering will skip cached files automatically
3. For training, add `training.resume_from` override in Cell 8

### Quick smoke test (~10 min on T4)
Set `N_STRUCTURES=50`, `MAX_EPOCHS=3`, `BATCH_SIZE=1`, `N_CONFORMERS=5` in Cell 4.